# 01 — Open Agenda Data Exploration
**Purpose:** Explore the "Deciding for Paris" agenda API, understand 
the data structure, validate field quality, and produce a clean 
filtered dataset ready for vectorization.

**Agenda:** Deciding for Paris (UID: 82290100)  
**Region:** Paris, Île-de-France  
**Date filter:** Events from the past 12 months + upcoming  
**Output:** `data/raw/events_paris.json`

**Author:** Hope  
**Date:** 2026-03-23

## 1. Imports and environment validation

In [1]:

import json
import time
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
import os
import requests
import pandas as pd

load_dotenv()

api_key = os.getenv("OPENAGENDA_API_KEY")
assert api_key is not None, "OPENAGENDA_API_KEY not found in .env"

print("Environment loaded successfully.")
print(f"API key loaded: {'*' * 20}{api_key[-4:]}")

Environment loaded successfully.
API key loaded: ********************a309


## 1.1 Configuration

In [2]:
AGENDA_UID  = "82290100"
BASE_URL    = f"https://api.openagenda.com/v2/agendas/{AGENDA_UID}/events"
OUTPUT_PATH = Path("../data/raw/events_paris.json")
LOCATION    = "Paris"
LIMIT       = 100
MAX_AGE_DAYS = 365

print("Configuration set:")
print(f"  Agenda UID  : {AGENDA_UID}")
print(f"  Location    : {LOCATION}")
print(f"  Output path : {OUTPUT_PATH}")
print(f"  Page limit  : {LIMIT}")
print(f"  Max age days: {MAX_AGE_DAYS}")

Configuration set:
  Agenda UID  : 82290100
  Location    : Paris
  Output path : ../data/raw/events_paris.json
  Page limit  : 100
  Max age days: 365


## 1.2 Single record exploration

In [3]:
# Fetch one record to confirm API access and inspect raw field structure

response = requests.get(BASE_URL, params={"key": api_key, "limit": 1})
assert response.status_code == 200, f"API error: {response.status_code} — {response.json()}"

sample = response.json()

print("Total events available:", sample["total"])
print("\n--- Field names (1 event) ---")
pprint(list(sample["events"][0].keys()))

Total events available: 9651

--- Field names (1 event) ---
['image',
 'featured',
 'attendanceMode',
 'keywords',
 'dateRange',
 'imageCredits',
 'originAgenda',
 'description',
 'evenements-lies-au-calendrier',
 'title',
 'onlineAccessLink',
 'uid',
 'lastTiming',
 'types-devenement',
 'firstTiming',
 'location',
 'categories',
 'slug',
 'status',
 'nextTiming']


## 1.3 Inspect the full raw structure of one event:

In [4]:
raw_event = sample["events"][0]
print(json.dumps(raw_event, indent=2, ensure_ascii=False))

{
  "image": {
    "filename": "a620461875854cdb83fba312f956d304.base.image.jpg",
    "size": {
      "width": 700,
      "height": 394
    },
    "variants": [
      {
        "filename": "a620461875854cdb83fba312f956d304.full.image.jpg",
        "size": {
          "width": 1920,
          "height": 1080
        },
        "type": "full"
      },
      {
        "filename": "a620461875854cdb83fba312f956d304.thumb.image.jpg",
        "size": {
          "width": 200,
          "height": 200
        },
        "type": "thumbnail"
      }
    ],
    "base": "https://cdn.openagenda.com/main/"
  },
  "featured": false,
  "attendanceMode": 1,
  "keywords": {},
  "dateRange": {
    "ar": "الإثنين ٢٣ مارس, 12:45",
    "de": "Montag 23 März, 12:45",
    "en": "Monday 23 March, 12:45",
    "it": "Lunedì 23 marzo, 12:45",
    "fr": "Lundi 23 mars, 12h45",
    "es": "Lunes 23 marzo, 12:45"
  },
  "imageCredits": "Collège des Bernardins",
  "originAgenda": {
    "image": "agenda75354844.jpg",
   

## 1.4 Revised Field Selection for Paris

In [5]:
COLS = [
    "uid",
    "title.fr",
    "description.fr",
    "location.city",
    "location.address",
    "location.name",
    "firstTiming.begin",
    "lastTiming.end",
    "dateRange.fr",
    "slug",
    "attendanceMode",
    "categories",
    "types-devenement"
]
print('Revised columns for Paris:')
pprint(COLS)

Revised columns for Paris:
['uid',
 'title.fr',
 'description.fr',
 'location.city',
 'location.address',
 'location.name',
 'firstTiming.begin',
 'lastTiming.end',
 'dateRange.fr',
 'slug',
 'attendanceMode',
 'categories',
 'types-devenement']


## 1.5 Full pagination fetch

In [6]:
all_events = []
cursor     = None
page       = 1

while True:
    print(f"Fetching page {page}...")

    params = {
        "key"        : api_key,
        "limit"      : LIMIT,
        "relative[0]": "passed",
        "relative[1]": "current",
        "relative[2]": "upcoming",
    }

    if cursor:
        params["after[0]"] = cursor[0]
        params["after[1]"] = cursor[1]
        params["after[2]"] = cursor[2]
        params["after[3]"] = cursor[3]

    response = requests.get(BASE_URL, params=params)
    assert response.status_code == 200, \
        f"API error: {response.status_code} — {response.json()}"

    data   = response.json()
    events = data.get("events", [])
    all_events.extend(events)

    print(f"  → {len(events)} fetched | cumulative: {len(all_events)}")

    if len(events) < LIMIT:
        print("Last page reached.")
        break

    cursor = data.get("after")
    page  += 1
    time.sleep(0.5)

print(f"\nTotal events fetched: {len(all_events)}")
assert len(all_events) > 0, "No events fetched"

Fetching page 1...
  → 100 fetched | cumulative: 100
Fetching page 2...
  → 100 fetched | cumulative: 200
Fetching page 3...
  → 100 fetched | cumulative: 300
Fetching page 4...
  → 100 fetched | cumulative: 400
Fetching page 5...
  → 100 fetched | cumulative: 500
Fetching page 6...
  → 100 fetched | cumulative: 600
Fetching page 7...
  → 100 fetched | cumulative: 700
Fetching page 8...
  → 100 fetched | cumulative: 800
Fetching page 9...
  → 100 fetched | cumulative: 900
Fetching page 10...
  → 100 fetched | cumulative: 1000
Fetching page 11...
  → 100 fetched | cumulative: 1100
Fetching page 12...
  → 100 fetched | cumulative: 1200
Fetching page 13...
  → 100 fetched | cumulative: 1300
Fetching page 14...
  → 100 fetched | cumulative: 1400
Fetching page 15...
  → 100 fetched | cumulative: 1500
Fetching page 16...
  → 100 fetched | cumulative: 1600
Fetching page 17...
  → 100 fetched | cumulative: 1700
Fetching page 18...
  → 100 fetched | cumulative: 1800
Fetching page 19...
  → 100 

## 1.6 Flatten and inspect full schema

In [7]:
df_raw = pd.json_normalize(all_events)

print(f"Shape (raw): {df_raw.shape}")
print(f"\nAll columns ({len(df_raw.columns)}):")
pprint(df_raw.columns.tolist())

Shape (raw): (9651, 53)

All columns (53):
['featured',
 'attendanceMode',
 'imageCredits',
 'evenements-lies-au-calendrier',
 'onlineAccessLink',
 'uid',
 'types-devenement',
 'categories',
 'slug',
 'status',
 'image.filename',
 'image.size.width',
 'image.size.height',
 'image.variants',
 'image.base',
 'dateRange.ar',
 'dateRange.de',
 'dateRange.en',
 'dateRange.it',
 'dateRange.fr',
 'dateRange.es',
 'originAgenda.image',
 'originAgenda.uid',
 'originAgenda.title',
 'description.fr',
 'title.fr',
 'lastTiming.end',
 'lastTiming.begin',
 'firstTiming.end',
 'firstTiming.begin',
 'location.address',
 'location.city',
 'location.latitude',
 'location.name',
 'location.longitude',
 'nextTiming.begin',
 'nextTiming.end',
 'keywords.fr',
 'image',
 'keywords.en',
 'description.en',
 'title.en',
 'nextTiming',
 'description.oc',
 'title.oc',
 'image.originalName',
 'image.extension',
 'description.de',
 'description.it',
 'description.es',
 'title.de',
 'title.it',
 'title.es']


## 1.7 Date Filter

In [8]:
df_raw["firstTiming.begin"] = pd.to_datetime(
    df_raw["firstTiming.begin"], utc=True
)

cutoff   = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=MAX_AGE_DAYS)
df_dated = df_raw[df_raw["firstTiming.begin"] >= cutoff].copy()

print(f"Before filter : {len(df_raw)} events")
print(f"After filter  : {len(df_dated)} events")
print(f"Dropped       : {len(df_raw) - len(df_dated)} events")
print(f"Date range    : {df_dated['firstTiming.begin'].min()} "
      f"→ {df_dated['firstTiming.begin'].max()}")

Before filter : 9651 events
After filter  : 1746 events
Dropped       : 7905 events
Date range    : 2025-03-23 11:00:00+00:00 → 2027-06-20 15:00:00+00:00


## 1.8 Refine date filter with upper bound

In [9]:
upper_bound = pd.Timestamp.now(tz="UTC") + pd.Timedelta(days=365)

df_dated = df_raw[
    (df_raw["firstTiming.begin"] >= cutoff) &
    (df_raw["firstTiming.begin"] <= upper_bound)
].copy()

print(f"After refined filter : {len(df_dated)} events")
print(f"Date range           : {df_dated['firstTiming.begin'].min()} "
      f"→ {df_dated['firstTiming.begin'].max()}")

After refined filter : 1742 events
Date range           : 2025-03-23 11:00:00+00:00 → 2027-03-21 16:00:00+00:00


## 1.9 Select relevant fields

In [10]:
COLS = [
    "uid",
    "title.fr",
    "description.fr",
    "keywords.fr",
    "location.city",
    "location.address",
    "location.name",
    "firstTiming.begin",
    "lastTiming.end",
    "dateRange.fr",
    "slug",
    "attendanceMode",
    "categories",
    "types-devenement"
]

df_selected = df_dated[COLS].copy()

print(f"Shape: {df_selected.shape}")
print(f"\nNull counts:\n{df_selected.isnull().sum()}")

Shape: (1742, 14)

Null counts:
uid                     0
title.fr                2
description.fr          2
keywords.fr          1544
location.city           9
location.address        0
location.name           0
firstTiming.begin       0
lastTiming.end          0
dateRange.fr            0
slug                    0
attendanceMode          0
categories              0
types-devenement        0
dtype: int64


## 2.1 Applying Cleaning Actions

In [11]:
# Data cleaning

# Action 1 — Drop keywords.fr (89% null — unreliable)
COLS_FINAL = [col for col in COLS if col != "keywords.fr"]
df_clean = df_selected[COLS_FINAL].copy()

# Action 2 — Drop rows with null title or description
before = len(df_clean)
df_clean = df_clean.dropna(subset=["title.fr", "description.fr"])
print(f"Dropped {before - len(df_clean)} rows with null title/description")

# Action 3 — Fix null location.city
df_clean["location.city"] = df_clean.apply(
    lambda row: "En ligne" if row["attendanceMode"] == 2
    else ("Non renseigné" if pd.isnull(row["location.city"])
    else row["location.city"]),
    axis=1
)
df_clean["location.city"] = df_clean["location.city"].replace(
    "75015 Paris", "Paris"
)
# Verify
assert df_clean["title.fr"].isnull().sum() == 0
assert df_clean["description.fr"].isnull().sum() == 0
assert df_clean["location.city"].isnull().sum() == 0

print(f"\nFinal shape: {df_clean.shape}")
print(f"\nNull counts:\n{df_clean.isnull().sum()}")

Dropped 2 rows with null title/description

Final shape: (1740, 13)

Null counts:
uid                  0
title.fr             0
description.fr       0
location.city        0
location.address     0
location.name        0
firstTiming.begin    0
lastTiming.end       0
dateRange.fr         0
slug                 0
attendanceMode       0
categories           0
types-devenement     0
dtype: int64


## 2.2 Preview and city Distribution

In [12]:
# Final dataset preview and distribution analysis

print("=== City Distribution ===")
print(df_clean["location.city"].value_counts().head(10))

print("\n=== Attendance Mode ===")
print(df_clean["attendanceMode"].value_counts())

print("\n=== Description length stats ===")
df_clean["desc_length"] = df_clean["description.fr"].str.len()
print(df_clean["desc_length"].describe())

print("\n=== Sample events ===")
df_clean[["title.fr", "location.city", "dateRange.fr", "description.fr"]].head(5)

=== City Distribution ===
location.city
Paris               1708
Non renseigné          9
Gif-sur-Yvette         7
En ligne               3
La Ferté-Imbault       2
Besançon               2
Vézelay                2
Épernon                1
Vanves                 1
Guesnain               1
Name: count, dtype: int64

=== Attendance Mode ===
attendanceMode
1    1718
3      19
2       3
Name: count, dtype: int64

=== Description length stats ===
count    1740.000000
mean       75.717816
std        60.805521
min         1.000000
25%        31.000000
50%        50.000000
75%       113.000000
max       200.000000
Name: desc_length, dtype: float64

=== Sample events ===


,title.fr,location.city,dateRange.fr,description.fr
0,Diderot : Jacques le Fataliste et son maître -...,Paris,"Lundi 23 mars, 12h45",Une pause déjeuner littéraire au cœur du quart...
1,"Jésus, vrai Dieu et vrai homme, comment y croi...",Paris,"12 janvier - 23 mars, les lundis","2 heures de cours par semaine le lundi, au cho..."
2,Collecte de dons pour les journées d’amitié,Paris,22 et 23 mars,Collecte de dons
3,Paris String Orchestra,Paris,"Lundi 23 mars, 20h00",Concert à La Madeleine à Paris.
4,Retrouver le sens et le goût des sacrements : ...,Paris,"6 janvier - 24 mars, certains mardis","2 heures de cours par semaine le mardi, au cho..."


## 2.3 Final fix and save

In [13]:
# Fix postal code in city field
df_clean["location.city"] = df_clean["location.city"].replace(
    "75015 Paris", "Paris"
)

# Drop helper column if added
if "desc_length" in df_clean.columns:
    df_clean = df_clean.drop(columns=["desc_length"])

# Final verification
assert df_clean["location.city"].isnull().sum() == 0
print(f"Final shape   : {df_clean.shape}")
print(f"Paris events  : {(df_clean['location.city'] == 'Paris').sum()}")

# Save to disk
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_json(
    OUTPUT_PATH,
    orient="records",
    force_ascii=False,
    indent=2
)
print(f"\nSaved {len(df_clean)} events → {OUTPUT_PATH}")

Final shape   : (1740, 13)
Paris events  : 1708

Saved 1740 events → ../data/raw/events_paris.json


In [15]:
import json
from pathlib import Path

with open("../data/raw/events_paris.json") as f:
    events = json.load(f)

print(type(events[0]["firstTiming.begin"]))
print(events[0]["firstTiming.begin"])

<class 'int'>
1774266300000


In [4]:
from graphviz import Digraph

dot = Digraph(
    name="rag_pipeline",
    graph_attr={
        "rankdir":  "LR",
        "bgcolor":  "transparent",
        "pad":      "0.4",
        "nodesep":  "0.6",
        "ranksep":  "0.8",
        "fontname": "Arial",
        "splines":  "ortho",
    },
    node_attr={
        "shape":     "box",
        "style":     "filled,rounded",
        "fontname":  "Arial",
        "fontsize":  "13",
        "penwidth":  "0.8",
        "width":     "1.6",
        "height":    "1.4",
        "margin":    "0.2,0.15",
    },
    edge_attr={
        "color":     "#BA7517",
        "penwidth":  "2.0",
        "arrowhead": "vee",
        "arrowsize": "0.8",
        "fontname":  "Arial",
        "fontsize":  "11",
        "fontcolor": "#5F5E5A",
    }
)

# Blue nodes
blue = {
    "fontcolor": "#0C447C",
    "fillcolor": "#E6F1FB",
    "color":     "#185FA5",
}

# Teal node (Embedding — transformation step)
teal = {
    "fontcolor": "#085041",
    "fillcolor": "#E1F5EE",
    "color":     "#0F6E56",
}

dot.node("open_agenda", label="Open Agenda\n\n9,651 events (variable)",    **blue)
dot.node("cleaning",    label="Cleaning\n\n1,740 retained",     **blue)
dot.node("embedding",   label="Embedding\n\nmistral-embed",     **teal)
dot.node("faiss",       label="FAISS index\n\n1,024 dims",      **blue)
dot.node("mistral",     label="Mistral LLM\n\nGeneration",      **blue)

dot.edge("open_agenda", "cleaning")
dot.edge("cleaning",    "embedding")
dot.edge("embedding",   "faiss")
dot.edge("faiss",       "mistral")

dot.render("rag_pipeline", format="png", cleanup=True)
dot.render("rag_pipeline", format="svg", cleanup=True)

print(dot.source)

digraph rag_pipeline {
	graph [bgcolor=transparent fontname=Arial nodesep=0.6 pad=0.4 rankdir=LR ranksep=0.8 splines=ortho]
	node [fontname=Arial fontsize=13 height=1.4 margin="0.2,0.15" penwidth=0.8 shape=box style="filled,rounded" width=1.6]
	edge [arrowhead=vee arrowsize=0.8 color="#BA7517" fontcolor="#5F5E5A" fontname=Arial fontsize=11 penwidth=2.0]
	open_agenda [label="Open Agenda

9,651 events (variable)" color="#185FA5" fillcolor="#E6F1FB" fontcolor="#0C447C"]
	cleaning [label="Cleaning

1,740 retained" color="#185FA5" fillcolor="#E6F1FB" fontcolor="#0C447C"]
	embedding [label="Embedding

mistral-embed" color="#0F6E56" fillcolor="#E1F5EE" fontcolor="#085041"]
	faiss [label="FAISS index

1,024 dims" color="#185FA5" fillcolor="#E6F1FB" fontcolor="#0C447C"]
	mistral [label="Mistral LLM

Generation" color="#185FA5" fillcolor="#E6F1FB" fontcolor="#0C447C"]
	open_agenda -> cleaning
	cleaning -> embedding
	embedding -> faiss
	faiss -> mistral
}



In [5]:
from graphviz import Digraph

dot = Digraph(
    name="inference_pipeline",
    graph_attr={
        "rankdir":  "LR",
        "bgcolor":  "transparent",
        "pad":      "0.4",
        "nodesep":  "0.5",
        "ranksep":  "0.7",
        "fontname": "Arial",
        "splines":  "ortho",
    },
    node_attr={
        "shape":     "box",
        "style":     "filled,rounded",
        "fontname":  "Arial",
        "fontsize":  "12",
        "fillcolor": "#FFFFFF",
        "color":     "#D3D1C7",
        "fontcolor": "#185FA5",
        "penwidth":  "0.8",
        "width":     "1.7",
        "height":    "1.2",
        "margin":    "0.2,0.2",
    },
    edge_attr={
        "color":     "#378ADD",
        "penwidth":  "1.5",
        "arrowhead": "vee",
        "arrowsize": "0.7",
    }
)

steps = [
    ("step1", "1\n\nRequete utilisateur"),
    ("step2", "2\n\nEmbedding requete"),
    ("step3", "3\n\nRecherche FAISS (top 5)"),
    ("step4", "4\n\nInjection contexte"),
    ("step5", "5\n\nReponse Mistral"),
    ("step6", "6\n\nAffichage + sources"),
]

for name, label in steps:
    dot.node(name, label=label)

for i in range(len(steps) - 1):
    dot.edge(steps[i][0], steps[i + 1][0])

dot.render("inference_pipeline", format="png", cleanup=True)
dot.render("inference_pipeline", format="svg", cleanup=True)

print(dot.source)

digraph inference_pipeline {
	graph [bgcolor=transparent fontname=Arial nodesep=0.5 pad=0.4 rankdir=LR ranksep=0.7 splines=ortho]
	node [color="#D3D1C7" fillcolor="#FFFFFF" fontcolor="#185FA5" fontname=Arial fontsize=12 height=1.2 margin="0.2,0.2" penwidth=0.8 shape=box style="filled,rounded" width=1.7]
	edge [arrowhead=vee arrowsize=0.7 color="#378ADD" penwidth=1.5]
	step1 [label="1

Requete utilisateur"]
	step2 [label="2

Embedding requete"]
	step3 [label="3

Recherche FAISS (top 5)"]
	step4 [label="4

Injection contexte"]
	step5 [label="5

Reponse Mistral"]
	step6 [label="6

Affichage + sources"]
	step1 -> step2
	step2 -> step3
	step3 -> step4
	step4 -> step5
	step5 -> step6
}



In [6]:
from graphviz import Digraph

dot = Digraph(
    name="pipeline_automatise",
    graph_attr={
        "rankdir":  "TB",
        "bgcolor":  "transparent",
        "pad":      "0.4",
        "nodesep":  "0.5",
        "ranksep":  "0.6",
        "fontname": "Arial",
        "splines":  "ortho",
    },
    node_attr={
        "shape":    "box",
        "style":    "filled,rounded",
        "fontname": "Arial",
        "fontsize": "12",
        "penwidth": "0.8",
        "width":    "3.4",
        "height":   "0.9",
        "margin":   "0.25,0.15",
    },
    edge_attr={
        "penwidth":  "1.5",
        "arrowhead": "vee",
        "arrowsize": "0.8",
        "fontname":  "Arial",
        "fontsize":  "10",
        "fontcolor": "#5F5E5A",
    }
)

# Ingestion
dot.node(
    "fetch",
    label="Recuperation des evenements\nAPI Open Agenda, paginee",
    fillcolor="#E6F1FB", color="#185FA5", fontcolor="#0C447C"
)

# Traitement
dot.node(
    "embed",
    label="Decoupage et vectorisation\nEmbeddings Mistral, limitation de debit",
    fillcolor="#E1F5EE", color="#0F6E56", fontcolor="#085041"
)

dot.node(
    "index",
    label="Construction de l'index vectoriel\nFAISS + stockage des metadonnees",
    fillcolor="#E1F5EE", color="#0F6E56", fontcolor="#085041"
)

# Verrou de qualite
dot.node(
    "test",
    shape="diamond",
    label="Controle qualite des donnees\n24 verifications automatisees",
    fillcolor="#FAEEDA", color="#854F0B", fontcolor="#633806",
    width="2.6", height="1.6", margin="0.15,0.1"
)

# Chemin d'echec
dot.node(
    "fail",
    label="Pipeline interrompu\ncorriger les donnees, relancer",
    fillcolor="#FCEBEB", color="#A32D2D", fontcolor="#791F1F",
    width="2.4"
)

# Evaluation et deploiement
dot.node(
    "eval",
    label="Evaluation de la qualite\n30 questions annotees, RAGAS",
    fillcolor="#E6F1FB", color="#185FA5", fontcolor="#0C447C"
)

dot.node(
    "run",
    label="Deploiement de l'assistant\nInterface chatbot en direct",
    fillcolor="#E6F1FB", color="#185FA5", fontcolor="#0C447C"
)

# Connexions
dot.edge("fetch", "embed")
dot.edge("embed", "index")
dot.edge("index", "test")

dot.edge("test", "fail", label="echec", color="#A32D2D", fontcolor="#A32D2D")
dot.edge("test", "eval", label="reussi", color="#0F6E56", fontcolor="#0F6E56")

dot.edge("eval", "run")

with dot.subgraph() as s:
    s.attr(rank="same")
    s.node("fail")

dot.render("pipeline_automatise", format="png", cleanup=True)
dot.render("pipeline_automatise", format="svg", cleanup=True)

print(dot.source)

digraph pipeline_automatise {
	graph [bgcolor=transparent fontname=Arial nodesep=0.5 pad=0.4 rankdir=TB ranksep=0.6 splines=ortho]
	node [fontname=Arial fontsize=12 height=0.9 margin="0.25,0.15" penwidth=0.8 shape=box style="filled,rounded" width=3.4]
	edge [arrowhead=vee arrowsize=0.8 fontcolor="#5F5E5A" fontname=Arial fontsize=10 penwidth=1.5]
	fetch [label="Recuperation des evenements
API Open Agenda, paginee" color="#185FA5" fillcolor="#E6F1FB" fontcolor="#0C447C"]
	embed [label="Decoupage et vectorisation
Embeddings Mistral, limitation de debit" color="#0F6E56" fillcolor="#E1F5EE" fontcolor="#085041"]
	index [label="Construction de l'index vectoriel
FAISS + stockage des metadonnees" color="#0F6E56" fillcolor="#E1F5EE" fontcolor="#085041"]
	test [label="Controle qualite des donnees
24 verifications automatisees" color="#854F0B" fillcolor="#FAEEDA" fontcolor="#633806" height=1.6 margin="0.15,0.1" shape=diamond width=2.6]
	fail [label="Pipeline interrompu
corriger les donnees, relanc